## DP1 patch diagnostic for one VIRCAM band

It does **one-band** checks, controlled by `TARGET_BAND` (default: `K`).

### Goals

For the selected VIRCAM band, it will:

1. build one **simple patch table** with:
   - `tract`
   - `patch`
   - `calexp_exists`
   - `forced_exists`
   - `forced_rows`

2. save that table to CSV

3. create only these three problem tables:
   - **missing coadd calexp**
   - **missing forced catalog**
   - **low-row existing forced catalogs** (`forced_rows < FORCED_MIN_ROWS`)

### Notes

- This notebook checks **only VIRCAM** `deepCoadd_calexp`, not ComCam.
- `forced_rows` is set to **0** when the forced catalog is missing, so the main table is easy to read.
- To run another band later, change only:

```python
TARGET_BAND = "K"
```


In [79]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from lsst.daf.butler import Butler


In [80]:
# ----------------------------
# User configuration
# ----------------------------

BUTLER_LOC = "../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data"

COLL_MEAS_FORCED   = "u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z"
COLL_CALEXP_VIRCAM = "u/ir-sare1/DRP/videoCoaddDetect"

SKYMAP = "lsst_cells_v1"
TARGET_BAND = "K"          # change later if needed: "Z", "Y", "J", "H", "K"
FORCED_MIN_ROWS = 4000

OUTDIR = Path("./data/patches")
OUTDIR.mkdir(parents=True, exist_ok=True)

print("BUTLER_LOC =", BUTLER_LOC)
print("TARGET_BAND =", TARGET_BAND)
print("FORCED_MIN_ROWS =", FORCED_MIN_ROWS)
print("Output directory =", OUTDIR.resolve())


BUTLER_LOC = ../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data
TARGET_BAND = K
FORCED_MIN_ROWS = 4000
Output directory = /rds/project/rds-rPTGgs6He74/Elham/lsst-ir-fusion/dmu5/dmu5_DP1/diagnostics/data/patches


In [81]:
# ----------------------------
# Patch list from your dataQuery
# ----------------------------

PATCHES_BY_TRACT = {
    4848: [60, 61, 70, 71, 72, 73, 80, 81, 82, 83, 90, 91, 92, 93],
    4849: [66, 67, 68, 69, 75, 76, 77, 78, 79, 85, 86, 87, 88, 89, 94, 95, 96, 97, 98, 99],
    5062: [0, 10, 20],
    5063: [0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25,
           26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 41, 42, 43, 44, 45, 46, 47, 48, 52, 53,
           54, 55, 56, 57],
    5064: [9, 19, 29, 39],
}

PATCH_TABLE = pd.DataFrame(
    [(tract, patch) for tract, patches in PATCHES_BY_TRACT.items() for patch in patches],
    columns=["tract", "patch"]
).sort_values(["tract", "patch"]).reset_index(drop=True)

print("Total requested patches =", len(PATCH_TABLE))
display(PATCH_TABLE.head(20))


Total requested patches = 91


,tract,patch
0,4848,60
1,4848,61
2,4848,70
3,4848,71
4,4848,72
5,4848,73
6,4848,80
7,4848,81
8,4848,82
9,4848,83


In [82]:
# ----------------------------
# Butler
# ----------------------------

butler = Butler(BUTLER_LOC)
registry = butler.registry

print("Butler initialised successfully.")


Butler initialised successfully.


In [83]:
# ----------------------------
# Helper functions
# ----------------------------

def get_single_ref(dataset_type, data_id, collections):
    refs = list(
        registry.queryDatasets(
            dataset_type,
            dataId=data_id,
            collections=collections,
            findFirst=True,
        )
    )
    return refs[0] if refs else None


def dataset_exists(dataset_type, data_id, collections):
    return get_single_ref(dataset_type, data_id, collections) is not None


def count_table_rows(dataset_type, data_id, collections):
    ref = get_single_ref(dataset_type, data_id, collections)
    if ref is None:
        return False, 0

    obj = butler.get(ref)

    # Most LSST tables support len(table)
    try:
        nrows = len(obj)
    except Exception:
        # fallback if needed
        try:
            nrows = len(obj.asAstropy())
        except Exception:
            nrows = np.nan

    return True, int(nrows) if pd.notna(nrows) else np.nan


In [84]:
# ----------------------------
# Build the main one-band patch table
# ----------------------------

rows = []

for _, rr in PATCH_TABLE.iterrows():
    tract = int(rr["tract"])
    patch = int(rr["patch"])

    data_id = {
        "skymap": SKYMAP,
        "tract": tract,
        "patch": patch,
        "band": TARGET_BAND,
    }

    calexp_exists = dataset_exists(
        "deepCoadd_calexp",
        data_id=data_id,
        collections=COLL_CALEXP_VIRCAM,
    )

    forced_exists, forced_rows = count_table_rows(
        "deepCoadd_forced_src",
        data_id=data_id,
        collections=COLL_MEAS_FORCED,
    )

    rows.append({
        "tract": tract,
        "patch": patch,
        "band": TARGET_BAND,
        "calexp_exists": bool(calexp_exists),
        "forced_exists": bool(forced_exists),
        "forced_rows": int(forced_rows) if pd.notna(forced_rows) else np.nan,
    })

band_df = pd.DataFrame(rows).sort_values(["tract", "patch"]).reset_index(drop=True)

print(f"Main {TARGET_BAND}-band table:")
print("Number of patches =", len(band_df))
display(band_df)


Main K-band table:
Number of patches = 91


,tract,patch,band,calexp_exists,forced_exists,forced_rows
0,4848,60,K,True,True,15170
1,4848,61,K,True,True,14613
2,4848,70,K,True,True,18063
3,4848,71,K,True,True,19028
4,4848,72,K,True,True,15973
...,...,...,...,...,...,...
86,5063,57,K,True,True,12600
87,5064,9,K,True,True,13065
88,5064,19,K,True,True,12409
89,5064,29,K,True,True,13528


In [85]:
# Save the main table
main_csv = OUTDIR / f"{TARGET_BAND}_band_patch_table.csv"
band_df.to_csv(main_csv, index=False)
print("Saved:", main_csv)


Saved: data/patches/K_band_patch_table.csv


In [86]:
# ----------------------------
# Problem tables for the selected band
# ----------------------------

missing_calexp_df = (
    band_df.loc[~band_df["calexp_exists"], ["tract", "patch", "band"]]
    .sort_values(["tract", "patch"])
    .reset_index(drop=True)
)

missing_forced_df = (
    band_df.loc[~band_df["forced_exists"], ["tract", "patch", "band", "forced_rows"]]
    .sort_values(["tract", "patch"])
    .reset_index(drop=True)
)

low_row_existing_df = (
    band_df.loc[
        band_df["forced_exists"] & (band_df["forced_rows"] < FORCED_MIN_ROWS),
        ["tract", "patch", "band", "forced_rows"]
    ]
    .sort_values(["tract", "patch"])
    .reset_index(drop=True)
)

print(f"Missing {TARGET_BAND}-band calexp patches =", len(missing_calexp_df))
display(missing_calexp_df)

print(f"Missing {TARGET_BAND}-band forced patches =", len(missing_forced_df))
display(missing_forced_df)

print(f"Low-row existing {TARGET_BAND}-band forced patches (< {FORCED_MIN_ROWS}) =", len(low_row_existing_df))
display(low_row_existing_df)


Missing K-band calexp patches = 0


,tract,patch,band


Missing K-band forced patches = 5


,tract,patch,band,forced_rows
0,4848,90,K,0
1,4849,98,K,0
2,5063,5,K,0
3,5063,13,K,0
4,5063,24,K,0


Low-row existing K-band forced patches (< 4000) = 4


,tract,patch,band,forced_rows
0,4849,66,K,3941
1,5062,10,K,339
2,5063,8,K,2856
3,5063,55,K,216


In [87]:
# Save the three problem tables
missing_calexp_csv = OUTDIR / f"{TARGET_BAND}_band_missing_calexp.csv"
missing_forced_csv = OUTDIR / f"{TARGET_BAND}_band_missing_forced.csv"
low_row_existing_csv = OUTDIR / f"{TARGET_BAND}_band_low_row_existing.csv"

missing_calexp_df.to_csv(missing_calexp_csv, index=False)
missing_forced_df.to_csv(missing_forced_csv, index=False)
low_row_existing_df.to_csv(low_row_existing_csv, index=False)

print("Saved:", missing_calexp_csv)
print("Saved:", missing_forced_csv)
print("Saved:", low_row_existing_csv)


Saved: data/patches/K_band_missing_calexp.csv
Saved: data/patches/K_band_missing_forced.csv
Saved: data/patches/K_band_low_row_existing.csv


In [88]:
# ----------------------------
# Very short summary
# ----------------------------

summary_df = pd.DataFrame([{
    "band": TARGET_BAND,
    "n_requested_patches": len(band_df),
    "n_missing_calexp": len(missing_calexp_df),
    "n_missing_forced": len(missing_forced_df),
    "n_low_row_existing_forced": len(low_row_existing_df),
}])

display(summary_df)

summary_csv = OUTDIR / f"{TARGET_BAND}_band_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)


,band,n_requested_patches,n_missing_calexp,n_missing_forced,n_low_row_existing_forced
0,K,91,0,5,4


Saved: data/patches/K_band_summary.csv


In [90]:
# show only tract/patch + rows, for very fast reading
simple_rows_view = band_df[["tract", "patch", "forced_rows"]].copy()
display(simple_rows_view)


,tract,patch,forced_rows
0,4848,60,15170
1,4848,61,14613
2,4848,70,18063
3,4848,71,19028
4,4848,72,15973
...,...,...,...
86,5063,57,12600
87,5064,9,13065
88,5064,19,12409
89,5064,29,13528
